In [1]:
import pandas as pd
import numpy as np

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')


# Define mapping for admission types
emergency_types = ['DIRECT EMER.', 'EW EMER.', 'URGENT']
normal_types = ['AMBULATORY OBSERVATION', 'DIRECT OBSERVATION', 'ELECTIVE', 
                'EU OBSERVATION', 'OBSERVATION ADMIT', 'SURGICAL SAME DAY ADMISSION']

# Create a new column 'admission_category' based on the mapping
admissions['admission_category'] = admissions['admission_type'].apply(
    lambda x: 'EMERGENCY' if x in emergency_types else 'NORMAL'
)

# Check if mapping worked correctly
print(admissions['admission_category'].value_counts())

# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'diagnosis_description', 'icd_code']]
procedures = procedures[['subject_id', 'hadm_id', 'procedure_description', 'icd_code']]

# ------------------------- STEP 3: COMPUTE AGE AT EVENTS & MORTALITY -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'dod']]

# Merge patient data into all event tables
admissions = admissions.merge(patients, on='subject_id', how='left')
icustays = icustays.merge(patients, on='subject_id', how='left')
diagnoses = diagnoses.merge(patients, on='subject_id', how='left')
procedures = procedures.merge(patients, on='subject_id', how='left')
prescriptions = prescriptions.merge(patients, on='subject_id', how='left')

# Convert date columns
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
patients['dod'] = pd.to_datetime(patients['dod'], errors='coerce')

# Compute age at event time
admissions['age_at_event'] = admissions['anchor_age'] + (admissions['admittime'].dt.year - admissions['anchor_year'])
prescriptions['age_at_event'] = prescriptions['anchor_age'] + (prescriptions['starttime'].dt.year - prescriptions['anchor_year'])

# Assign age at death for deceased patients
patients['age_at_death'] = patients['anchor_age'] + (patients['dod'].dt.year - patients['anchor_year'])
patients['death_flag'] = patients['dod'].notna().astype(int)

# Merge mortality data into admissions
admissions = admissions.merge(patients[['subject_id', 'death_flag', 'age_at_death']], on='subject_id', how='left')

##### Adding Died During Visit variable 
# Convert discharge time to datetime format
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'], errors='coerce')

# Determine if the patient died during this admission
admissions['died_during_visit'] = (
    (admissions['death_flag'] == 1) & 
    (admissions['dod'] >= admissions['admittime']) & 
    (admissions['dod'] <= admissions['dischtime'])
).astype(int)

##### Creating how many days until the next hospitalization
# Sort admissions by patient ID and admission time
admissions = admissions.sort_values(by=['subject_id', 'admittime'])

# Get the next admission time per patient
admissions['next_admittime'] = admissions.groupby('subject_id')['admittime'].shift(-1)

# Compute the number of days until the next visit
admissions['days_until_next_visit'] = (admissions['next_admittime'] - admissions['dischtime']).dt.days

# Compute the number of minutes until the next visit
admissions['mins_until_next_visit'] = (admissions['next_admittime'] - admissions['dischtime']).dt.total_seconds() / 60

# Fill NaN values with -1 (for last visit of each patient)
admissions['days_until_next_visit'].fillna(-1, inplace=True)

admissions['mins_until_next_visit'].fillna(-1, inplace=True)

##### How many days from the last visit to the death?
# Get the last discharge time per patient
last_discharge = admissions.groupby('subject_id')['dischtime'].max().reset_index()
last_discharge.rename(columns={'dischtime': 'last_dischtime'}, inplace=True)

# Merge last discharge time into patients data
patients = patients.merge(last_discharge, on='subject_id', how='left')

# Calculate days from last visit to death (if patient died after last visit)
patients['days_from_last_visit_to_death'] = (patients['dod'] - patients['last_dischtime']).dt.days

# If patient didn't die, set the value to NaN or -1
patients.loc[patients['death_flag'] == 0, 'days_from_last_visit_to_death'] = np.nan

admissions = admissions.merge(
    patients[['subject_id', 'days_from_last_visit_to_death']], 
    on='subject_id', 
    how='left'
)

# ------------------------- STEP 4: BUILD TABULAR DATASET -------------------------

# Select main features
features = admissions[['subject_id',
                       'hadm_id',
                       'admittime',
                       'dischtime',
                       'age_at_event',
                       'gender',
                       'admission_type',
                       'admission_category',
                       'discharge_location',
                       'death_flag',
                       'age_at_death',
                       'died_during_visit',
                       'days_from_last_visit_to_death',
                       'days_until_next_visit',
                       'mins_until_next_visit'
                       ]]

# ICU stays: Number of ICU admissions and length of stay per hospitalization
icu_summary = icustays.groupby('hadm_id').agg(
    icu_admissions=('stay_id', 'count'),
    icu_days=('intime', lambda x: (x.max() - x.min()).days)
).reset_index()

# Diagnoses: Count number of diagnoses per hospitalization
diagnosis_summary = diagnoses.groupby('hadm_id').agg(
    num_diagnoses=('icd_code', 'count'),
    diagnosis_list=('diagnosis_description', lambda x: list(x.unique()))
).reset_index()

# Procedures: Count number of procedures per hospitalization
procedure_summary = procedures.groupby('hadm_id').agg(
    num_procedures=('icd_code', 'count'),
    procedure_list=('procedure_description', lambda x: list(x.unique()))
).reset_index()

# Medications: Count number of prescribed drugs per hospitalization
med_summary = prescriptions.groupby('hadm_id').agg(
    num_medications=('drug', 'count'),
    medication_list=('drug', lambda x: list(x.unique()))
).reset_index()

# Merge all features into a single dataset
dataset = features.merge(icu_summary, on='hadm_id', how='left')
dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left')
dataset = dataset.merge(procedure_summary, on='hadm_id', how='left')
dataset = dataset.merge(med_summary, on='hadm_id', how='left')

# Fill missing values with 0
dataset.fillna({'icu_admissions': 0, 'icu_days': 0, 'num_diagnoses': 0, 'num_procedures': 0, 'num_medications': 0}, inplace=True)
dataset.fillna({'diagnosis_list': 'None', 'procedure_list': 'None', 'medication_list': 'None'}, inplace=True)

/tmp/ipykernel_4045/746790821.py:19: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")


admission_category
NORMAL       291667
EMERGENCY    254361
Name: count, dtype: int64


In [2]:
admissions['subject_id'].nunique()

223452

In [3]:
dataset[dataset['subject_id']==11530780][['subject_id','admittime', 'dischtime', 'days_until_next_visit']]

,subject_id,admittime,dischtime,days_until_next_visit
82093,11530780,2150-04-15 03:25:00,2150-04-16 16:20:00,-1.0
82094,11530780,2150-04-15 22:48:00,2150-04-16 00:20:00,80.0
82095,11530780,2150-07-05 23:06:00,2150-07-08 16:20:00,-1.0


In [4]:
admissions_sorted = dataset.sort_values(['subject_id', 'admittime'])

In [5]:
# Check for overlaps (current visit starts before previous ends)
admissions_sorted['overlaps'] = (
    admissions_sorted.groupby('subject_id')['admittime'].shift(-1) < admissions_sorted['dischtime']
)

In [6]:
admissions_sorted[admissions_sorted['overlaps']==True]

,subject_id,hadm_id,admittime,dischtime,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,...,mins_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list,overlaps
3928,10076639,23684554,2135-12-22 22:53:00,2135-12-23 19:19:00,60,F,EU OBSERVATION,NORMAL,NaN,0,...,-120.0,0.0,0.0,1.0,"[Schizoaffective disorder, unspecified]",3.0,"[Electrocardiogram, Computerized axial tomogra...",0.0,None,True
25110,10482555,24141586,2146-10-02 00:11:00,2146-10-12 11:00:00,21,F,OBSERVATION ADMIT,NORMAL,HOME,0,...,-13844.0,0.0,0.0,3.0,"[Major depressive disorder, recurrent severe w...",0.0,None,29.0,"[Diazepam, HydrOXYzine, LORazepam, Sertraline,...",True
36926,10693757,23954106,2163-04-20 00:00:00,2163-04-22 15:45:00,37,F,URGENT,EMERGENCY,HOME,0,...,-2879.0,0.0,0.0,2.0,[Other mental disorders complicating the puerp...,0.0,None,10.0,"[LORazepam, Senna, Ibuprofen, Polyethylene Gly...",True
53749,11011076,27826883,2178-12-02 21:43:00,2178-12-03 09:15:00,82,M,DIRECT OBSERVATION,NORMAL,NaN,1,...,-120.0,0.0,0.0,6.0,[Other complications due to genitourinary devi...,0.0,None,0.0,None,True
71745,11339384,27383384,2187-09-16 04:36:00,2187-09-26 14:47:00,78,F,EW EMER.,EMERGENCY,SKILLED NURSING FACILITY,0,...,-317.0,1.0,0.0,39.0,"[Other amyloidosis, Acute on chronic diastolic...",0.0,None,106.0,"[Doxycycline Hyclate, Pantoprazole, Metoprolol...",True
75244,11402448,20437968,2138-09-17 17:54:00,2138-09-17 23:12:00,68,F,EU OBSERVATION,NORMAL,NaN,1,...,-59.0,0.0,0.0,11.0,"[Unspecified psychosis, Unspecified episodic m...",0.0,None,0.0,None,True
77609,11443713,22227792,2146-05-16 00:01:00,2146-05-17 23:44:00,87,F,EU OBSERVATION,NORMAL,NaN,1,...,-1413.0,0.0,0.0,10.0,"[Gouty arthropathy, unspecified, Atrial fibril...",0.0,None,0.0,None,True
82093,11530780,26728216,2150-04-15 03:25:00,2150-04-16 16:20:00,22,F,DIRECT OBSERVATION,NORMAL,NaN,0,...,-1052.0,0.0,0.0,4.0,[Other current conditions classifiable elsewhe...,0.0,None,4.0,"[Albuterol Inhaler, Prenatal Vitamins, Acetami...",True
83415,11553072,20836013,2163-03-22 00:07:00,2163-03-23 07:11:00,53,M,EU OBSERVATION,NORMAL,NaN,1,...,-1457.0,0.0,0.0,4.0,"[Alcohol abuse, unspecified, Unspecified essen...",0.0,None,0.0,None,True
85056,11582633,20212371,2133-05-29 22:57:00,2133-05-30 09:43:00,50,F,EU OBSERVATION,NORMAL,NaN,0,...,-8.0,0.0,0.0,1.0,"[Depressive disorder, not elsewhere classified]",0.0,None,0.0,None,True


In [7]:
TEST_df = admissions_sorted[admissions_sorted['subject_id']==10076639]

In [8]:
TEST_df

,subject_id,hadm_id,admittime,dischtime,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,...,mins_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list,overlaps
3928,10076639,23684554,2135-12-22 22:53:00,2135-12-23 19:19:00,60,F,EU OBSERVATION,NORMAL,NaN,0,...,-120.0,0.0,0.0,1.0,"[Schizoaffective disorder, unspecified]",3.0,"[Electrocardiogram, Computerized axial tomogra...",0.0,None,True
3929,10076639,23036582,2135-12-23 17:19:00,2136-01-01 15:30:00,60,F,EW EMER.,EMERGENCY,HOME HEALTH CARE,0,...,-1.0,0.0,0.0,5.0,"[Schizoaffective disorder, chronic with acute ...",0.0,None,32.0,"[Desonide 0.05% Cream, Risperidone (Disintegra...",False


In [9]:
TEST_df = TEST_df.sort_values(by=['subject_id', 'admittime'])
TEST_df = TEST_df.reset_index(drop=True)

In [10]:
TEST_df

,subject_id,hadm_id,admittime,dischtime,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,...,mins_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list,overlaps
0,10076639,23684554,2135-12-22 22:53:00,2135-12-23 19:19:00,60,F,EU OBSERVATION,NORMAL,NaN,0,...,-120.0,0.0,0.0,1.0,"[Schizoaffective disorder, unspecified]",3.0,"[Electrocardiogram, Computerized axial tomogra...",0.0,None,True
1,10076639,23036582,2135-12-23 17:19:00,2136-01-01 15:30:00,60,F,EW EMER.,EMERGENCY,HOME HEALTH CARE,0,...,-1.0,0.0,0.0,5.0,"[Schizoaffective disorder, chronic with acute ...",0.0,None,32.0,"[Desonide 0.05% Cream, Risperidone (Disintegra...",False


In [11]:
admissions_sorted['subject_id'].nunique()


223452

What can be done in this situation? We can try different approaches, starting from the concatenation in an unique visit of the overlapping visits for all 48 patients. Or, since there are just 48 patients with overlapping over 223452 (i.e., 0.02%) we decide to remove those patients from our dataset.

In [12]:
dataframe_no_overlap = admissions_sorted.groupby('subject_id').filter(lambda x: all(x['overlaps']==False))

In [13]:
dataframe_no_overlap['subject_id'].nunique() # no more overlaps

223404

In [14]:
dataframe_no_overlap[dataframe_no_overlap['mins_until_next_visit']==0]


,subject_id,hadm_id,admittime,dischtime,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,...,mins_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list,overlaps
185,10002930,28477649,2197-04-07 06:56:00,2197-04-08 19:37:00,52,F,EU OBSERVATION,NORMAL,NaN,1,...,0.0,0.0,0.0,7.0,"[Unspecified psychosis, Suicidal ideation, Uns...",0.0,None,0.0,None,False
970,10017035,29784303,2143-04-10 23:55:00,2143-04-13 11:38:00,25,M,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,4.0,[Poisoning by other opiates and related narcot...,0.0,None,0.0,None,False
1595,10029211,29967424,2133-08-10 19:42:00,2133-08-13 10:35:00,25,F,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,6.0,"[Unspecified mood [affective] disorder, Suicid...",0.0,None,0.0,None,False
1621,10029874,23069127,2180-11-13 18:27:00,2180-11-18 19:44:00,67,M,EW EMER.,EMERGENCY,PSYCH FACILITY,0,...,0.0,1.0,0.0,12.0,"[Acute alcoholic intoxication in alcoholism, u...",0.0,None,37.0,"[Sodium Chloride 0.9% Flush, FoLIC Acid, 0.9%...",False
2055,10038688,29585848,2175-06-10 04:43:00,2175-06-12 21:33:00,46,M,OBSERVATION ADMIT,NORMAL,PSYCH FACILITY,1,...,0.0,0.0,0.0,10.0,"[Alcohol withdrawal, Acidosis, Viral hepatitis...",1.0,[Alcohol detoxification],16.0,"[Sodium Chloride 0.9% Flush, Diazepam, Senna,...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545029,19983279,22656324,2127-12-01 01:15:00,2127-12-04 15:02:00,36,F,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,4.0,"[Manic episode, unspecified, Post-traumatic st...",0.0,None,0.0,None,False
545053,19983966,29065074,2137-07-11 22:21:00,2137-07-13 13:15:00,85,F,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,13.0,"[Major depressive disorder, single episode, un...",0.0,None,0.0,None,False
545132,19985349,21686305,2127-09-19 02:04:00,2127-09-19 03:54:00,72,M,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,3.0,"[Other and unspecified alcohol dependence, uns...",0.0,None,0.0,None,False
545623,19993214,24464076,2170-10-09 15:20:00,2170-10-09 20:10:00,40,M,EU OBSERVATION,NORMAL,NaN,0,...,0.0,0.0,0.0,5.0,"[Unspecified schizophrenia, unspecified, Unspe...",0.0,None,0.0,None,False


In [15]:
dataframe_no_overlap[dataframe_no_overlap['subject_id']==10002930][['subject_id', 'admittime', 'dischtime', 'days_until_next_visit', 'mins_until_next_visit']]

,subject_id,admittime,dischtime,days_until_next_visit,mins_until_next_visit
182,10002930,2193-08-05 06:18:00,2193-08-05 11:44:00,0.0,1.0
183,10002930,2193-08-05 11:45:00,2193-08-11 09:25:00,977.0,1407060.0
184,10002930,2196-04-14 12:25:00,2196-04-17 15:28:00,354.0,510688.0
185,10002930,2197-04-07 06:56:00,2197-04-08 19:37:00,0.0,0.0
186,10002930,2197-04-08 19:37:00,2197-04-15 12:01:00,1.0,2280.0
187,10002930,2197-04-17 02:01:00,2197-04-17 09:48:00,365.0,526190.0
188,10002930,2198-04-17 19:38:00,2198-04-22 16:02:00,0.0,15.0
189,10002930,2198-04-22 16:17:00,2198-05-04 13:20:00,289.0,416665.0
190,10002930,2199-02-17 21:45:00,2199-02-19 13:38:00,470.0,677765.0
191,10002930,2200-06-05 05:43:00,2200-06-05 10:26:00,252.0,363272.0


In [ ]:
# Let us load the landmark_df_evo dataset
landmark_df_evo = pd.read_csv("/root/MIMICIV/src/landmark_df_evo.csv")

In [2]:
import pandas as pd

In [3]:
landmark_df_evo_correct = pd.read_csv("/root/MIMICIV/src/landmark_df_evo_correct.csv")

In [ ]:
landmark_df_evo['subject_id'].nunique()

In [4]:
landmark_df_evo_correct['subject_id'].nunique()

223452

In [20]:
#landmark_df_evo.head()

In [20]:
dataframe_no_overlap_2_merge = dataframe_no_overlap[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'days_until_next_visit', 'mins_until_next_visit']].copy()
dataframe_no_overlap_2_merge.head()

,subject_id,hadm_id,admittime,dischtime,days_until_next_visit,mins_until_next_visit
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,50.0,72072.0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,25.0,37066.0
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,11.0,16189.0
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,-1.0,-1.0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,-1.0,-1.0


In [21]:
dataframe_no_overlap_2_merge['subject_id'].nunique()

223404

In [23]:
# Ensure your DataFrame is sorted by patient and landmark_visit
dataframe_no_overlap_2_merge = dataframe_no_overlap_2_merge.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

# Function to perform transformation
def transform_to_past_variable(group):
    group = group.sort_values('admittime').copy()
    # Shift the values down (future to past)
    group['days_since_last_visit'] = group['days_until_next_visit'].shift(1)
    # First landmark always has -1
    group['days_since_last_visit'].iloc[0] = -1
    # same reasoning for minutes
    group['mins_since_last_visit'] = group['mins_until_next_visit'].shift(1)
    # First landmark always has -1
    group['mins_since_last_visit'].iloc[0] = -1

    return group

# Apply the transformation per patient
dataframe_no_overlap_2_merge = dataframe_no_overlap_2_merge.groupby('subject_id').apply(transform_to_past_variable).reset_index(drop=True)


In [24]:
dataframe_no_overlap_2_merge.head()


,subject_id,hadm_id,admittime,dischtime,days_until_next_visit,mins_until_next_visit,days_since_last_visit,mins_since_last_visit
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,50.0,72072.0,-1.0,-1.0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,25.0,37066.0,50.0,72072.0
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,11.0,16189.0,25.0,37066.0
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,-1.0,-1.0,11.0,16189.0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,-1.0,-1.0,-1.0,-1.0


In [26]:
dataframe_no_overlap_2_merge.to_csv("/root/MIMICIV/src/dataframe_no_overlap_2_merge.csv", index=False)

In [1]:
import pandas as pd

In [3]:
dataframe_no_overlap_2_merge = pd.read_csv("/root/MIMICIV/src/dataframe_no_overlap_2_merge.csv")
landmark_df_evo_correct = pd.read_csv("/root/MIMICIV/src/landmark_df_evo_correct.csv")


In [ ]:
landmark_df_evo['subject_id'].nunique()  # Check unique patients in landmark dataset<

Landmark_df_evo (the dataset on which we are basing all the analysis) looses 55_037 patients from admissions (after removong the 48 patients). 
We need to investigate starting from the dataset at the base of the landmark_evo dataset what happens.

In [ ]:
mimic_tab_death_visit_evo = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit_evo.csv")


In [22]:
mimic_tab_death_visit_evo['subject_id'].nunique()

223452

Dataset mimiciv_clinical_dataset_tabular_death_visit_evo.csv has the right number of patients. Then that's good. So, something happend in creating landmark_evo dataset.
Please, refer to the code **check_data_creation_evo.ipynb** for this.

We produced the **landmark_df_evo_correct** which has exactly the same amount of patients of the admissions (it could have been a problem in saving).

In [5]:
landmark_df_evo_correct_no_overlap = landmark_df_evo_correct.merge(dataframe_no_overlap_2_merge, on=['subject_id', 'hadm_id'], how='inner')

In [6]:
landmark_df_evo_correct_no_overlap

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit,admittime,dischtime,days_until_next_visit,mins_until_next_visit,days_since_last_visit,mins_since_last_visit
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,{1: ['Percutaneous abdominal drainage']},{1: ['40 mg of Furosemide in TAB through PO/NG...,0,-1.0,2180-05-06 22:23:00,2180-05-07 17:15:00,50.0,72072.0,-1.0,-1.0
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,50.0,2180-06-26 18:27:00,2180-06-27 18:49:00,25.0,37066.0,50.0,72072.0
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,25.0,2180-07-23 12:35:00,2180-07-25 17:55:00,11.0,16189.0,25.0,37066.0
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,"{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['40 mg of Furosemide in TAB through PO/NG...,0,11.0,2180-08-05 23:44:00,2180-08-07 17:50:00,-1.0,-1.0,11.0,16189.0
4,10000068,25022803,NORMAL,1,19,F,1,0,NaN,"Alcohol abuse, unspecified",...,{1: ['Other nonoperative respiratory measureme...,{1: ['']},0,-1.0,2160-03-03 23:16:00,2160-03-04 06:26:00,-1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544973,19999828,29734428,EMERGENCY,1,46,F,2,0,Magnesium Sulfate\nTopiramate (Topamax)\nBag\n...,Disruption of external operation (surgical) wo...,...,{1: ['Replacement of Abdomen Skin with Autolog...,{1: ['2 gm of Magnesium Sulfate in BAG through...,0,-1.0,2147-07-18 16:23:00,2147-08-04 18:10:00,522.0,753034.0,-1.0,-1.0
544974,19999828,25744818,EMERGENCY,2,48,F,2,0,Magnesium Sulfate\nTopiramate (Topamax)\nBag\n...,Disruption of external operation (surgical) wo...,...,{1: ['Replacement of Abdomen Skin with Autolog...,{1: ['2 gm of Magnesium Sulfate in BAG through...,0,522.0,2149-01-08 16:44:00,2149-01-18 17:00:00,-1.0,-1.0,522.0,753034.0
544975,19999840,26071774,EMERGENCY,1,58,M,2,1,NS\nLeVETiracetam\nMetoprolol Tartrate\nAcetam...,"Cerebral artery occlusion, unspecified with ce...",...,{1: ['Magnetic resonance imaging of brain and ...,"{1: ['500 mL of NS in mL through IV', '1500 mg...",1,-1.0,2164-07-25 00:27:00,2164-07-28 12:15:00,44.0,63452.0,-1.0,-1.0
544976,19999840,21033226,EMERGENCY,2,58,M,2,1,NS\nLeVETiracetam\nMetoprolol Tartrate\nAcetam...,"Cerebral artery occlusion, unspecified with ce...",...,{1: ['Magnetic resonance imaging of brain and ...,"{1: ['500 mL of NS in mL through IV', '1500 mg...",1,44.0,2164-09-10 13:47:00,2164-09-17 13:42:00,-1.0,-1.0,44.0,63452.0


In [7]:
landmark_df_evo_correct_no_overlap['subject_id'].nunique()

223404

Now let's test whether the full prompting scheme works here. There is a problem: few patients are readmitted almost at the same time of dischargment of the previous visit. What to do? The easy approach is to force also for them 1 day-delay (in this case 1 day will be the smallest time unit). Another possible approach would be to go to minutes/seconds in such cases. Howevere there are also cases where the patient gets admitted exactly at the same time of discharge. 

In [8]:
landmark_df_evo_correct_no_overlap['days_since_last_visit'] = landmark_df_evo_correct_no_overlap['days_since_last_visit'].replace(0, 1)

In [9]:
import numpy as np

In [10]:
landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate'] = landmark_df_evo_correct_no_overlap.groupby('subject_id')['days_since_last_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])
landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate_sum'] = landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])



In [11]:
landmark_df_evo_correct_no_overlap

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,gender_numeric,days_since_previous_visit,admittime,dischtime,days_until_next_visit,mins_until_next_visit,days_since_last_visit,mins_since_last_visit,days_since_last_visit_cumulate,days_since_last_visit_cumulate_sum
0,10000032,22595853,EMERGENCY,1,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,0,-1.0,2180-05-06 22:23:00,2180-05-07 17:15:00,50.0,72072.0,-1.0,-1.0,[-1.0],[]
1,10000032,22841357,EMERGENCY,2,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,0,50.0,2180-06-26 18:27:00,2180-06-27 18:49:00,25.0,37066.0,50.0,72072.0,"[-1.0, 50.0]",[50.0]
2,10000032,29079034,EMERGENCY,3,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,0,25.0,2180-07-23 12:35:00,2180-07-25 17:55:00,11.0,16189.0,25.0,37066.0,"[-1.0, 50.0, 25.0]","[75.0, 25.0]"
3,10000032,25742920,EMERGENCY,4,52,F,4,1,Furosemide\nIpratropium Bromide Neb\nPotassium...,Portal hypertension\nOther ascites\nCirrhosis ...,...,0,11.0,2180-08-05 23:44:00,2180-08-07 17:50:00,-1.0,-1.0,11.0,16189.0,"[-1.0, 50.0, 25.0, 11.0]","[86.0, 36.0, 11.0]"
4,10000068,25022803,NORMAL,1,19,F,1,0,NaN,"Alcohol abuse, unspecified",...,0,-1.0,2160-03-03 23:16:00,2160-03-04 06:26:00,-1.0,-1.0,-1.0,-1.0,[-1.0],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544973,19999828,29734428,EMERGENCY,1,46,F,2,0,Magnesium Sulfate\nTopiramate (Topamax)\nBag\n...,Disruption of external operation (surgical) wo...,...,0,-1.0,2147-07-18 16:23:00,2147-08-04 18:10:00,522.0,753034.0,-1.0,-1.0,[-1.0],[]
544974,19999828,25744818,EMERGENCY,2,48,F,2,0,Magnesium Sulfate\nTopiramate (Topamax)\nBag\n...,Disruption of external operation (surgical) wo...,...,0,522.0,2149-01-08 16:44:00,2149-01-18 17:00:00,-1.0,-1.0,522.0,753034.0,"[-1.0, 522.0]",[522.0]
544975,19999840,26071774,EMERGENCY,1,58,M,2,1,NS\nLeVETiracetam\nMetoprolol Tartrate\nAcetam...,"Cerebral artery occlusion, unspecified with ce...",...,1,-1.0,2164-07-25 00:27:00,2164-07-28 12:15:00,44.0,63452.0,-1.0,-1.0,[-1.0],[]
544976,19999840,21033226,EMERGENCY,2,58,M,2,1,NS\nLeVETiracetam\nMetoprolol Tartrate\nAcetam...,"Cerebral artery occlusion, unspecified with ce...",...,1,44.0,2164-09-10 13:47:00,2164-09-17 13:42:00,-1.0,-1.0,44.0,63452.0,"[-1.0, 44.0]",[44.0]


In [12]:
visit_counts = landmark_df_evo_correct_no_overlap['subject_id'].value_counts()
selected_patients = visit_counts[visit_counts == 3].index
landmark_df_evo_correct_no_overlap_selected = landmark_df_evo_correct_no_overlap[landmark_df_evo_correct_no_overlap['subject_id'].isin(selected_patients)].copy()

In [13]:
landmark_df_evo_correct_no_overlap_selected

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,gender_numeric,days_since_previous_visit,admittime,dischtime,days_until_next_visit,mins_until_next_visit,days_since_last_visit,mins_since_last_visit,days_since_last_visit_cumulate,days_since_last_visit_cumulate_sum
22,10000826,20032235,EMERGENCY,1,32,F,3,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,0,-1.0,2146-12-05 19:07:00,2146-12-12 16:30:00,6.0,8709.0,-1.0,-1.0,[-1.0],[]
23,10000826,21086876,EMERGENCY,2,32,F,3,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,0,6.0,2146-12-18 17:39:00,2146-12-24 19:55:00,6.0,8928.0,6.0,8709.0,"[-1.0, 6.0]",[6.0]
24,10000826,28289260,EMERGENCY,3,32,F,3,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,0,6.0,2146-12-31 00:43:00,2147-01-02 17:45:00,-1.0,-1.0,6.0,8928.0,"[-1.0, 6.0, 6.0]","[12.0, 6.0]"
44,10001186,24906418,EMERGENCY,1,46,F,3,0,Sodium Chloride 0.9% Flush\nOndansetron\nGuai...,Disruption of external operation (surgical) wo...,...,0,-1.0,2188-09-24 15:33:00,2188-09-26 14:40:00,23.0,34115.0,-1.0,-1.0,[-1.0],[]
45,10001186,24016413,NORMAL,2,46,F,3,0,Sodium Chloride 0.9% Flush\nOndansetron\nGuai...,Disruption of external operation (surgical) wo...,...,0,23.0,2188-10-20 07:15:00,2188-10-21 15:05:00,635.0,915370.0,23.0,34115.0,"[-1.0, 23.0]",[23.0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544914,19999043,24799384,NORMAL,2,35,F,3,0,OxyCODONE (Immediate Release)\nFerrous Sulfate...,(Induced) termination of pregnancy with other ...,...,0,50.0,2164-12-18 22:42:00,2164-12-24 12:45:00,161.0,231991.0,50.0,72279.0,"[-1.0, 50.0]",[50.0]
544915,19999043,21756272,NORMAL,3,36,F,3,0,OxyCODONE (Immediate Release)\nFerrous Sulfate...,(Induced) termination of pregnancy with other ...,...,0,161.0,2165-06-03 15:16:00,2165-06-04 12:55:00,-1.0,-1.0,161.0,231991.0,"[-1.0, 50.0, 161.0]","[211.0, 161.0]"
544926,19999287,25875727,NORMAL,1,71,F,3,1,SW\nMagnesium Sulfate\nPantoprazole\nHYDROmorp...,"Malignant neoplasm of upper lobe, bronchus or ...",...,0,-1.0,2191-12-29 07:15:00,2192-01-11 19:00:00,2022.0,2912189.0,-1.0,-1.0,[-1.0],[]
544927,19999287,22997012,EMERGENCY,2,77,F,3,1,SW\nMagnesium Sulfate\nPantoprazole\nHYDROmorp...,"Malignant neoplasm of upper lobe, bronchus or ...",...,0,2022.0,2197-07-26 03:29:00,2197-07-31 14:00:00,3.0,4738.0,2022.0,2912189.0,"[-1.0, 2022.0]",[2022.0]


In [14]:
import ast
import random
from collections import Counter

def full_narrative(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    narrative += f"Today is the {current_visit} visit.\n"

    max_visit = int(row['landmark_visit'])

    # if row['days_since_previous_visit'] != -1:
    #     narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    narrative += "\nDiagnosis history:"
    for past_visit, diags in reversed(list(ast.literal_eval(row['diag_per_visit']).items())):
        if past_visit == max_visit:
            narrative += f"\nToday: {'; '.join(diags)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][int(past_visit) -1])} days ago: {'; '.join(diags)}."

    narrative += "\nPrescriptions history:"
    for past_visit, meds in reversed(ast.literal_eval(row['meds_per_visit']).items()):
        if past_visit ==  max_visit:
            narrative += f"\nToday: {'; '.join(meds)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][past_visit-1])} days ago: {'; '.join(meds)}."

    narrative += "\nProcedures history:"
    for past_visit, proc in reversed(ast.literal_eval(row['proc_per_visit']).items()):
        if past_visit == max_visit:
            narrative += f"\nToday: {'; '.join(proc)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][past_visit-1])} days ago: {'; '.join(proc)}."
    
    return narrative

def full_narrative_no_time_rnd(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient?\n"

    # Diagnosi
    all_diags = []
    diag_dict = ast.literal_eval(row['diag_per_visit'])
    for diags in diag_dict.values():
        all_diags.extend(diags)
    random.shuffle(all_diags)
    narrative += "\nDiagnosis history:\n" + '; '.join(all_diags) + "."

    # Prescrizioni
    all_meds = []
    meds_dict = ast.literal_eval(row['meds_per_visit'])
    for meds in meds_dict.values():
        all_meds.extend(meds)
    random.shuffle(all_meds)
    narrative += "\nPrescriptions history:\n" + '; '.join(all_meds) + "."

    # Procedure
    all_procs = []
    proc_dict = ast.literal_eval(row['proc_per_visit'])
    for procs in proc_dict.values():
        all_procs.extend(procs)
    random.shuffle(all_procs)
    narrative += "\nProcedures history:\n" + '; '.join(all_procs) + "."

    return narrative

def no_narrative_prompt(row):
    narrative = "Variables: "
    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" {row['diag_text']}"
    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" {row['med_text']}"
    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" {row['proc_text']}"

    return narrative

def compact_narrative_prompt(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {'; '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {'; '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {'; '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {'; '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {'; '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {'; '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative


In [15]:
train_df = landmark_df_evo_correct_no_overlap_selected[landmark_df_evo_correct_no_overlap_selected['landmark_visit'] == 3].copy()

In [16]:
train_df.columns

Index(['subject_id', 'hadm_id', 'admission_category', 'landmark_visit',
       'age_at_landmark', 'gender', 'num_total_visits', 'death_in_90days',
       'med_text', 'diag_text', 'proc_text', 'dose_text', 'new_medications',
       'new_diagnoses', 'new_procedures', 'new_dose', 'no_more_diagnoses',
       'no_more_medications', 'no_more_procedures', 'no_more_dose',
       'meds_per_visit', 'diag_per_visit', 'proc_per_visit', 'dose_per_visit',
       'gender_numeric', 'days_since_previous_visit', 'admittime', 'dischtime',
       'days_until_next_visit', 'mins_until_next_visit',
       'days_since_last_visit', 'mins_since_last_visit',
       'days_since_last_visit_cumulate', 'days_since_last_visit_cumulate_sum'],
      dtype='object')

In [17]:
problem_subjects = train_df[train_df.apply(lambda x: len(x['days_since_last_visit_cumulate_sum'])==0, axis=1)]['subject_id'].reset_index(drop=True).tolist()

In [18]:
landmark_df_evo_correct_no_overlap_selected[landmark_df_evo_correct_no_overlap_selected['subject_id'].isin(problem_subjects)]

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,gender_numeric,days_since_previous_visit,admittime,dischtime,days_until_next_visit,mins_until_next_visit,days_since_last_visit,mins_since_last_visit,days_since_last_visit_cumulate,days_since_last_visit_cumulate_sum


In [19]:
train_texts_full = train_df.apply(full_narrative, axis=1).tolist()
train_texts_full_no_time_rnd = train_df.apply(full_narrative_no_time_rnd, axis=1).tolist()
train_texts_no_narrative = train_df.apply(no_narrative_prompt, axis = 1).tolist()
train_texts_compact = train_df.apply(compact_narrative_prompt, axis = 1).tolist()

In [20]:
print(train_texts_full[12])

You are a Doctor.
What is the probability of death in the next 90 days from today for this 78-year-old M patient?
Today is the 3 visit.

Diagnosis history:
Today: Aortic valve disorders; Mobitz (type) II atrioventricular block; Unspecified pleural effusion; Other B-complex deficiencies; Atrial fibrillation; Thrombocytopenia, unspecified; Pulmonary collapse; Cardiac complications, not elsewhere classified; Coronary atherosclerosis of native coronary artery; Unspecified essential hypertension; Osteoarthrosis, unspecified whether generalized or localized, hand; Depressive disorder, not elsewhere classified; Other and unspecified hyperlipidemia; Iron deficiency anemia, unspecified; Orthostatic hypotension; Unspecified glaucoma; Atherosclerosis of aorta; Degeneration of cervical intervertebral disc; Degeneration of lumbar or lumbosacral intervertebral disc; Surgical operation with implant of artificial internal device causing abnormal patient reaction, or later complication,without mention 

In [21]:
print(train_texts_full_no_time_rnd[12])

You are a Doctor.
What is the probability of death in the next 90 days from today for this 78-year-old M patient?

Diagnosis history:
Osteoarthrosis, unspecified whether generalized or localized, hand; Coronary atherosclerosis of native coronary artery; Degeneration of cervical intervertebral disc; Orthostatic hypotension; Unspecified glaucoma; Percutaneous transluminal coronary angioplasty status; Unspecified glaucoma; Orthostatic hypotension; Other and unspecified hyperlipidemia; Other and unspecified angina pectoris; Personal history of other malignant neoplasm of skin; Depressive disorder, not elsewhere classified; Personal history of malignant neoplasm of prostate; Personal history of tobacco use; Unspecified hereditary and idiopathic peripheral neuropathy; Aortic valve disorders; Personal history of malignant neoplasm of bronchus and lung; Lumbosacral spondylosis without myelopathy; Dysphonia; Coronary atherosclerosis of native coronary artery; Other B-complex deficiencies; Atria

In [22]:
print(train_texts_no_narrative[12])

Variables:  Malignant neoplasm of lower lobe, bronchus or lung
Mobitz (type) II atrioventricular block
Loss of weight
Body Mass Index between 19-24, adult
Coronary atherosclerosis of native coronary artery
Percutaneous transluminal coronary angioplasty status
Personal history of malignant neoplasm of prostate
Aortic valve disorders
Cerebral atherosclerosis
Anemia, unspecified
Unspecified essential hypertension
Other and unspecified hyperlipidemia
Impaired glucose tolerance test (oral)
Other and unspecified alcohol dependence, in remission
Osteoarthrosis, unspecified whether generalized or localized, hand
Cervical spondylosis without myelopathy
Lumbosacral spondylosis without myelopathy
Depressive disorder, not elsewhere classified
Unspecified glaucoma
Unspecified hereditary and idiopathic peripheral neuropathy
Dysphonia
Personal history of tobacco use
Retention of urine, unspecified
Orthostatic hypotension
Atrial fibrillation
Hypopotassemia
Disorders of phosphorus metabolism
Disorders 

In [23]:
print(train_texts_compact[12])

You are a Doctor.
What is the probability of death in the next 90 days for 78-year-old M patient?
Visit number 3 - EMERGENCY 
Last visit happened 521.0 days ago.
DIAGNOSIS HISTORY:
Chronic diagnoses: Mobitz (type) II atrioventricular block; Coronary atherosclerosis of native coronary artery; Percutaneous transluminal coronary angioplasty status; Personal history of malignant neoplasm of prostate; Aortic valve disorders; Unspecified essential hypertension; Other and unspecified hyperlipidemia; Osteoarthrosis, unspecified whether generalized or localized, hand; Depressive disorder, not elsewhere classified; Unspecified glaucoma; Personal history of tobacco use; Orthostatic hypotension; Atrial fibrillation.
New diagnoses in this visit: Unspecified pleural effusion; Other B-complex deficiencies; Thrombocytopenia, unspecified; Pulmonary collapse; Cardiac complications, not elsewhere classified; Iron deficiency anemia, unspecified; Atherosclerosis of aorta; Degeneration of cervical intervert

In [24]:
train_df.iloc[12]

subject_id                                                                     10005348
hadm_id                                                                        25239799
admission_category                                                            EMERGENCY
landmark_visit                                                                        3
age_at_landmark                                                                      78
gender                                                                                M
num_total_visits                                                                      3
death_in_90days                                                                       0
med_text                              Ketorolac\nOxycoDONE (Immediate Release) \nHYD...
diag_text                             Malignant neoplasm of lower lobe, bronchus or ...
proc_text                             15 mg of Ketorolac in VIAL through IV\n5-10 mg...
dose_text                       

In [25]:
#ast.literal_eval(train_df['diag_text'])
row = train_df.iloc[12]
list(ast.literal_eval(row['diag_per_visit']).items())

[(1,
  ['Malignant neoplasm of lower lobe, bronchus or lung',
   'Mobitz (type) II atrioventricular block',
   'Loss of weight',
   'Body Mass Index between 19-24, adult',
   'Coronary atherosclerosis of native coronary artery',
   'Percutaneous transluminal coronary angioplasty status',
   'Personal history of malignant neoplasm of prostate',
   'Aortic valve disorders',
   'Cerebral atherosclerosis',
   'Anemia, unspecified',
   'Unspecified essential hypertension',
   'Other and unspecified hyperlipidemia',
   'Impaired glucose tolerance test (oral)',
   'Other and unspecified alcohol dependence, in remission',
   'Osteoarthrosis, unspecified whether generalized or localized, hand',
   'Cervical spondylosis without myelopathy',
   'Lumbosacral spondylosis without myelopathy',
   'Depressive disorder, not elsewhere classified',
   'Unspecified glaucoma',
   'Unspecified hereditary and idiopathic peripheral neuropathy',
   'Dysphonia',
   'Personal history of tobacco use',
   'Retenti

In [1]:
# Let's see lenghts:
len_full = len(train_texts_full[12])
len_full_no_time_rnd = len(train_texts_full_no_time_rnd[12])
len_no_narrative = len(train_texts_no_narrative[12])
len_compact = len(train_texts_compact[12])
print(len_full, len_full_no_time_rnd, len_no_narrative, len_compact)


NameError: name 'train_texts_full' is not defined